# N-gram Sentence Prediction

This notebook builds a very simple sentence generator from the quote dataset.

It uses:
- 4-gram if the context is available
- trigram if 4-gram is not available
- bigram if trigram is not available
- unigram as the final fallback

It also handles unknown words with an `<unk>` token.

# Setup and Imports

Load the libraries needed for tokenization, preprocessing, and model training.

In [1]:
import nltk
import numpy as np
import pandas as pd
import re
from collections import Counter, defaultdict
from nltk.tokenize import word_tokenize

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
print("Setup completed successfully.")

Setup completed successfully.


# Load the Dataset

Read the CSV file and build the training corpus used by the language model.

In [2]:
# Option A (default): Quote CSV. Leave this block uncommented to use the CSV dataset.
dataset_path = "qoute_dataset.csv"

try:
    df = pd.read_csv(dataset_path)
    corpus_text = " ".join(df.astype(str).values.flatten())
    print(f"Loaded '{dataset_path}' successfully with {len(corpus_text)} characters.")
except Exception as e:
    print(f"Error reading dataset: {e}")

# Option B: Plain text file (e.g. dracula_corpus.txt).
# To use the text file instead, comment out the CSV block above and uncomment the block below.
# dataset_path = "dracula_corpus.txt"
# try:
#     with open(dataset_path, encoding="utf-8") as f:
#         corpus_text = f.read()
#     print(f"Loaded '{dataset_path}' successfully with {len(corpus_text)} characters.")
# except Exception as e:
#     print(f"Error reading dataset: {e}")

def preprocess_corpus(corpus_text, min_freq=2):
    """Tokenizes corpus text, lowers case, and replaces low-frequency words with <unk>."""
    tokens = word_tokenize(corpus_text.lower())

    freqs = Counter(tokens)

    processed_tokens = []
    vocab = set()

    for token in tokens:
        if freqs[token] >= min_freq:
            processed_tokens.append(token)
            vocab.add(token)
        else:
            processed_tokens.append("<unk>")

    vocab.add("<unk>")
    return processed_tokens, vocab


def process_user_input(input_text, vocab):
    """Converts user input tokens to <unk> if they are missing from vocabulary."""
    tokens = word_tokenize(input_text.lower())
    return [token if token in vocab else "<unk>" for token in tokens]

Loaded 'qoute_dataset.csv' successfully with 520966 characters.


# Preprocess the Corpus

Tokenize the dataset, normalize words, and replace rare tokens with `<unk>`.

In [3]:
class LaplaceNGramModel:
    def __init__(self, corpus_tokens, vocab, max_n=4):
        self.max_n = max_n
        self.vocab = sorted(vocab)
        self.vocab_set = set(vocab)
        self.vocab_size = len(vocab)
        self.unigram_counts = Counter(corpus_tokens)
        self.ngram_counts = {
            2: defaultdict(Counter),
            3: defaultdict(Counter),
            4: defaultdict(Counter),
        }

        for n in range(2, self.max_n + 1):
            for i in range(len(corpus_tokens) - n + 1):
                context = tuple(corpus_tokens[i : i + n - 1])
                next_word = corpus_tokens[i + n - 1]
                self.ngram_counts[n][context][next_word] += 1

    def get_ngram_prob(self, context, word):
        """Calculates P(word | context) using Laplace (+1) smoothing."""
        n = len(context) + 1
        context_counts = self.ngram_counts[n].get(tuple(context), Counter())
        context_total = sum(context_counts.values())
        count_context_word = context_counts[word]

        return (count_context_word + 1) / (context_total + self.vocab_size)

    def _sample_from_context(self, context):
        probs = np.array([self.get_ngram_prob(context, word) for word in self.vocab])
        probs = probs / np.sum(probs)
        return np.random.choice(self.vocab, p=probs)

    def predict_next_word(self, history):
        """Predicts the next word with 4-gram, trigram, bigram, then unigram backoff."""
        history = list(history)

        for order in range(min(self.max_n - 1, len(history)), 0, -1):
            context = tuple(history[-order:])
            if context in self.ngram_counts[order + 1]:
                return self._sample_from_context(context)

        probs = np.array([
            (self.unigram_counts[word] + 1) / (len(self.vocab_set) + sum(self.unigram_counts.values()))
            for word in self.vocab
        ])
        probs = probs / np.sum(probs)
        return np.random.choice(self.vocab, p=probs)

    def generate_sentence(self, prompt_text, max_words=15):
        """Generates a sentence using 4-gram, trigram, and bigram backoff."""
        processed_prompt = process_user_input(prompt_text, self.vocab_set)

        if not processed_prompt:
            sentence = ["<unk>"]
        else:
            sentence = processed_prompt.copy()

        for _ in range(max_words):
            next_word = self.predict_next_word(sentence)
            if next_word == "<unk>" and sentence[-1] == "<unk>":
                continue
            sentence.append(next_word)

            if next_word in [".", "!", "?"]:
                break

        return " ".join(sentence)

# Train the N-Gram Model

Build the Laplace-smoothed 4-gram, trigram, and bigram backoff model from the processed corpus.

In [4]:
corpus_tokens, vocab = preprocess_corpus(corpus_text, min_freq=2)

model = LaplaceNGramModel(corpus_tokens, vocab)

print("Model Training Complete")
print(f"Total Tokens Processed : {len(corpus_tokens)}")
print(f"Vocabulary Size (|V|)  : {len(vocab)}")
print("Sample Vocabulary      :", list(vocab)[:10])

Model Training Complete
Total Tokens Processed : 114661
Vocabulary Size (|V|)  : 4318
Sample Vocabulary      : ['planning', 'eliot', 'pamuk', 'praise', 'else', '؟', 'little', 'mad', 'edgar', 'promises']


# Generate a Sentence

Enter a prompt and let the model generate the next words.

In [5]:
user_prompt = input("Enter a starting phrase: ")

if user_prompt.strip():
    generated_text = model.generate_sentence(user_prompt, max_words=15)
    print("\nGeneration Output")
    print(f"User Input : {user_prompt}")
    print(f"Generated  : {generated_text}")
else:
    print("Please enter a non-empty phrase.")


Generation Output
User Input : if
Generated  : if they single minor revere pamuk quotation survived mysteries z think armchair jealousy filled glass ruined
